# LangChain - Generating Dynamic Questions / SQL query pairs

**Goal**: to generate a set of questions and SQL pairs to ask about the Chinook database.

The questions and SQL queries should have dynamic fields, example json output:
``` json
{
    "question": "What is the total revenue generated by sales support agent <employee_last_name>?",
    "answer": "SELECT e.FirstName, e.LastName, SUM(i.Total) as total_revenue FROM Employee e JOIN Customer c ON e.EmployeeId = c.SupportRepId JOIN Invoice i ON c.CustomerId = i.CustomerId WHERE e.LastName = '<employee_last_name>';"
}
```


## Pydantic

Lang Chain uses Pydantic classes to know the JSON format in which to output results.

Below is a class for a question/SQL pair, and one for a collection of question/SQL pair.

In [1]:
from pydantic import BaseModel, Field
from typing import List

class QuestionSQLPair(BaseModel):
    question: str = Field(description="A synthetic user question about the data.")
    sql_query: str = Field(description="A valid SQL query to answer the question.")

# This is the one passed to the chain
class QuestionBatch(BaseModel):
    items: List[QuestionSQLPair] = Field(description="A list of question-query pairs")

## Prompt Template

Fields:
- **template_fields**: A string in csv format containing the entire list of potential dynamic fields.
- **schema**: The complete schema of the database about which we need to generate questions.
- **target_table**: This field will allow me to break down the queriying to specific tables.
- **question_number**: The number of questions to generate.

In [2]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
        """
        You are a SQL expert.
        Your goal is to generate synthetic and dynamic User Questions and corresponding dynamic SQL queries based on a given data schema.
        
        **Rules:**
        1. Ensure the SQL is syntactically correct for SQLite.
        2. Use the provided Dynamic Fields placeholders (e.g. `<artist_name>`) instead of hardcoded values (e.g. 'AC/DC').
        3. Do not invent new dynamic fields; only use the ones listed below.

        **The "Spotlight" Rule:**
        You will be given a specific **Target Table**.
        Your questions MUST revolve around the metrics and attributes of that specific table.
        * **Good:** If Target is 'Employee', ask "Which Employee generated the most sales?" (Uses joins, but about Employee).
        * **Bad:** If Target is 'Employee', do not ask "Which Track is the longest?" (Irrelevant to Employee).

        **Available Dynamic Fields:**
```
{template_fields}
```
        
        **Few-Shot Examples:**
        
        Example 1: Time-Bound Sales
        {{
            "Q": "How many individual tracks did <artist_name> sell between <start_date> and <end_date>?",
            "A": "SELECT COUNT(il.InvoiceLineId) as total_tracks_sold FROM InvoiceLine il JOIN Track t ON il.TrackId = t.TrackId JOIN Album a ON t.AlbumId = a.AlbumId JOIN Artist art ON a.ArtistId = art.ArtistId JOIN Invoice i ON il.InvoiceId = i.InvoiceId WHERE art.Name = '<artist_name>' AND i.InvoiceDate BETWEEN '<start_date>' AND '<end_date>';"
        }}

        Example 2: Regional Market Analysis
        {{
            "Q": "List all customers in <country_name> who have purchased <genre_name> music.",
            "A": "SELECT DISTINCT c.FirstName, c.LastName, c.Email FROM Customer c JOIN Invoice i ON c.CustomerId = i.CustomerId JOIN InvoiceLine il ON i.InvoiceId = il.InvoiceId JOIN Track t ON il.TrackId = t.TrackId JOIN Genre g ON t.GenreId = g.GenreId WHERE g.Name = '<genre_name>' AND i.BillingCountry = '<country_name>';"
        }}

        Example 3: Employee Revenue
        {{
            "Q": "What is the total revenue generated by sales support agent <employee_last_name>?",
            "A": "SELECT e.FirstName, e.LastName, SUM(i.Total) as total_revenue FROM Employee e JOIN Customer c ON e.EmployeeId = c.SupportRepId JOIN Invoice i ON c.CustomerId = i.CustomerId WHERE e.LastName = '<employee_last_name>';"
        }}
        """
    ),
    ("human",
        """
        **Full Schema (Context):**
        {schema}

        **Target Table (Focus):**
        {target_table}

        **Task:**
        Generate {question_number} distinct questions specifically about the '{target_table}'.
        """
    )
])

# Print to ensure formatting is correct.
print(prompt.format(template_fields="artist_name, the full name of the artist ...", schema="CREATE TABLE...", target_table="Artist", question_number="5")[:75] + "...")

System: 
        You are a SQL expert.
        Your goal is to generate syn...


## Loading The Template Fields

This is a collection of the dynamic template fields which the LLM may choose to include in the generated data.

In [3]:
from functions import load_template_fields

loaded_template_fields = load_template_fields()

print (loaded_template_fields[:75] + "...")

field_name,description,data_type,example_value
artist_name,The full name of...


## Loading the Schema
I'm going to use the Chinook database in sqlite format for this test. I'm going to load and provide the entire schema because the LLM needs to be aware of all of the tables and the relationship beteween each other.

In [4]:
from functions import load_chinook_schema

ch_schema = load_chinook_schema(db_path="./data/Chinook_Sqlite.sqlite")

print(ch_schema[:75] + "...")

CREATE TABLE [Album]
(
    [AlbumId] INTEGER  NOT NULL,
    [Title] NVARCHA...


## Initialize Model And Create Chain

I'm going to use Gemini pro for this example as I already have a key handy.

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

load_dotenv("./secret.env")

gk = os.getenv("GEMINI_API_KEY")

# Initialize Gemini
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro",
    temperature=0
)

# This is where the output is structured to the Pydantic class.
structured_llm = llm.with_structured_output(QuestionBatch)

# Create final chain
chain = prompt | structured_llm

# Generating Data

## Use Chain To Generate Data

In [6]:
result = chain.invoke({
    "template_fields": loaded_template_fields,
    "schema": ch_schema,
    "target_table": "Artist",
    "question_number": "5"
})

In [7]:
for item in result.items:
    print(f"Q: {item.question}")
    print(f"A: {item.sql_query}")
    print("---------")

Q: What is the total revenue generated from sales of music by the artist <artist_name>?
A: SELECT SUM(il.UnitPrice * il.Quantity) FROM InvoiceLine AS il JOIN Track AS t ON il.TrackId = t.TrackId JOIN Album AS a ON t.AlbumId = a.AlbumId JOIN Artist AS ar ON a.ArtistId = ar.ArtistId WHERE ar.Name = '<artist_name>'
---------
Q: Which genre is most associated with the artist <artist_name>, based on the number of tracks sold?
A: SELECT T3.Name FROM Artist AS T1 INNER JOIN Album AS T2 ON T1.ArtistId = T2.ArtistId INNER JOIN Track AS T4 ON T2.AlbumId = T4.AlbumId INNER JOIN Genre AS T3 ON T4.GenreId = T3.GenreId WHERE T1.Name = '<artist_name>' GROUP BY T3.Name ORDER BY COUNT(T4.TrackId) DESC LIMIT 1
---------
Q: List all artists who have produced music in the <genre_name> genre.
A: SELECT DISTINCT T1.Name FROM Artist AS T1 INNER JOIN Album AS T2 ON T1.ArtistId = T2.ArtistId INNER JOIN Track AS T3 ON T2.AlbumId = T3.AlbumId INNER JOIN Genre AS T4 ON T3.GenreId = T4.GenreId WHERE T4.Name = '<ge

## Execute SQL queries

In [16]:
from functions import fill_sql_template, execute_query

for item in result.items:
    # Replace dynamic fields with real values. Ex: <artist> -> AC/DC
    converted_sql_query = fill_sql_template(item.sql_query)
    query_res = execute_query(converted_sql_query, "./data/Chinook_Sqlite.sqlite")
    print("Question: " + item.question)
    print("Answer: " + str(query_res))
    print("----------")

Question: What is the total revenue generated from sales of music by the artist <artist_name>?
Answer: [(15.84,)]
----------
Question: Which genre is most associated with the artist <artist_name>, based on the number of tracks sold?
Answer: [('Rock',)]
----------
Question: List all artists who have produced music in the <genre_name> genre.
Answer: [('AC/DC',), ('Accept',), ('Aerosmith',), ('Alanis Morissette',), ('Alice In Chains',), ('Audioslave',), ('Led Zeppelin',), ('Frank Zappa & Captain Beefheart',), ('Queen',), ('Kiss',), ('David Coverdale',), ('Deep Purple',), ('Santana',), ('Creedence Clearwater Revival',), ('Def Leppard',), ('Faith No More',), ('Foo Fighters',), ("Guns N' Roses",), ('Iron Maiden',), ('Jamiroquai',), ('Jimi Hendrix',), ('Joe Satriani',), ('Lenny Kravitz',), ('Marillion',), ('Men At Work',), ('Nirvana',), ('O Terço',), ('Ozzy Osbourne',), ('Page & Plant',), ("Paul D'Ianno",), ('Pearl Jam',), ('Pink Floyd',), ('R.E.M.',), ('Raul Seixas',), ('Red Hot Chili Pepper